## Load the required packages

In [1]:
import anndata
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from scipy.io import mmread
import pandas as pd
import os
from pathlib import Path
from scipy.sparse import csr_matrix

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/Dengkw"

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

In [4]:
normalized_train_gene_expression = mmread(f"{base_dir}/normalized_gene_expression_train.mtx").tocsr()
normalized_test_gene_expression = mmread(f"{base_dir}/normalized_gene_expression_test.mtx").tocsr()
normalized_train_gene_expression, normalized_test_gene_expression

(<7105x13553 sparse matrix of type '<class 'numpy.float64'>'
 	with 7658760 stored elements in Compressed Sparse Row format>,
 <8715x13553 sparse matrix of type '<class 'numpy.float64'>'
 	with 15662727 stored elements in Compressed Sparse Row format>)

## Select common proteins

In [5]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

## Normalize reference raw protein expression

In [6]:
gmean_train = ((train_protein_expression.applymap(np.log1p).sum(axis=1)) / train_protein_expression.shape[1]).apply(np.exp)
train_protein_clr = (train_protein_expression.div(gmean_train, axis=0)).applymap(np.log1p)
train_protein_data = anndata.AnnData(X=train_protein_clr)
train_protein_clr

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGGTATCTC-2,0.163406,1.082708,0.883817,0.536529,0.163406,0.163406,0.000000,0.725174,2.597197,0.163406,...,0.725174,2.440030,0.163406,0.303824,1.298208,0.426930,1.141091,0.536529,0.635293,1.298208
AAACCCACATTCTGTT-2,0.410438,1.354729,0.156285,0.410438,0.612911,1.108534,0.612911,0.410438,2.049804,0.410438,...,0.700598,0.000000,0.291414,0.000000,0.990134,0.291414,1.051085,0.410438,0.516791,1.831612
AAACCCAGTAGCTTGT-2,0.647618,1.495538,0.436052,0.547420,0.167374,0.436052,0.547420,0.738684,2.566869,0.167374,...,0.647618,2.348917,0.436052,0.167374,0.738684,0.000000,0.547420,0.167374,0.547420,1.574044
AAACCCAGTATCAGCT-2,0.172565,1.976231,0.756165,0.561553,0.447915,0.000000,0.561553,0.663586,2.051626,0.319687,...,1.059000,0.561553,0.000000,0.172565,0.919002,0.663586,0.663586,0.663586,0.319687,1.949781
AAACCCAGTTTCTATC-2,0.000000,1.468071,0.656491,0.656491,0.315691,0.832570,0.170251,0.315691,1.937200,0.748401,...,0.832570,0.170251,0.656491,0.000000,1.280501,0.170251,0.170251,0.442638,0.910202,1.227544
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTGGGCTCT-2,0.326559,2.073349,1.139128,0.572323,0.572323,0.326559,0.456972,0.456972,1.943899,0.326559,...,0.769445,0.572323,0.572323,0.000000,1.309232,0.000000,0.456972,0.456972,0.572323,1.360059
TTTGTTGTCAAATAGG-2,0.180154,0.947483,1.845274,0.781345,0.581995,0.465119,0.332753,0.581995,2.785005,0.332753,...,1.154178,1.712036,0.332753,0.180154,0.581995,0.332753,0.781345,0.332753,0.180154,2.552883
TTTGTTGTCACGAGGA-2,0.809858,1.464852,0.712782,0.484766,0.000000,0.347739,0.347739,0.809858,2.035203,0.347739,...,0.484766,2.414899,0.484766,0.000000,1.124707,0.484766,0.809858,0.605259,0.484766,1.640671


In [7]:
train_data.X = normalized_train_gene_expression
test_data.X = normalized_test_gene_expression

In [8]:
feature_obs = train_data.obs
gs_obs = train_protein_data.obs
test_obs = test_data.obs
protein_names = np.array(train_protein_data.var.index)
pred_dimx = test_data.shape[0]
pred_dimy = train_protein_data.shape[1]

feature_obs["batch"] = 0   
gs_obs["batch"] = 0
batches = [0]   
batch_len = len(batches)

In [9]:
input_train = anndata.concat({"train": train_data, "test": test_data}, axis=0, join="outer", label="group", fill_value=0, index_unique="-")

## Perform SVD dimensionality reduction on gene expression

In [10]:
n_mod1, n_mod2, scale, alpha = 300, 70, 10, 0.2
embedder_mod1 = TruncatedSVD(n_components=n_mod1)   
mod1_pca = embedder_mod1.fit_transform(input_train.X).astype(np.float32)
train_matrix = mod1_pca[input_train.obs['group'] == 'train']   
test_matrix = mod1_pca[input_train.obs['group'] == 'test']
train_matrix, test_matrix
train_matrix, test_matrix, train_matrix.shape, test_matrix.shape

(array([[ 2.88421726e+01, -5.17366076e+00, -5.70368195e+00, ...,
         -2.63123780e-01,  1.52912408e-01, -4.42117870e-01],
        [ 2.61759815e+01,  4.92808676e+00, -6.80456829e+00, ...,
         -2.78107643e-01, -1.15240765e+00,  2.07774222e-01],
        [ 2.58486328e+01, -4.79225016e+00, -7.74890137e+00, ...,
         -1.12347929e-02,  4.34952155e-02,  3.94242436e-01],
        ...,
        [ 2.44593010e+01, -3.48722839e+00, -6.57880688e+00, ...,
          7.48041153e-01,  1.69177726e-01, -1.26296841e-02],
        [ 2.72148914e+01,  4.94584131e+00, -6.62053204e+00, ...,
         -2.49418885e-01,  1.53234964e-02, -1.84643850e-01],
        [ 2.80564518e+01, -5.13456488e+00, -7.90295553e+00, ...,
          1.09257534e-01, -2.56896377e-01, -8.74978304e-01]], dtype=float32),
 array([[ 4.24270630e+01, -9.45506859e+00,  1.20453012e+00, ...,
          5.36056161e-01, -8.92247081e-01, -1.65131319e+00],
        [ 3.75674019e+01,  7.62884378e+00,  1.65656197e+00, ...,
          1.06776071e+0

## Perform SVD dimensionality reduction on protein expression (if >70 proteins)

In [11]:
if n_mod2 is not None and n_mod2 < train_protein_data.shape[1]:
    embedder_mod2 = TruncatedSVD(n_components=n_mod2)
    train_gs = embedder_mod2.fit_transform(train_protein_data.X).astype(np.float32)   
else:
    train_gs = train_protein_data.to_df().values.astype(np.float32)
train_gs, train_gs.shape

(array([[ 1.2470725e+01,  5.0920482e+00,  1.4242281e+00, ...,
         -1.9413579e-01, -4.4897515e-01,  9.8325625e-02],
        [ 1.3166884e+01, -3.9695418e+00, -6.9413811e-01, ...,
          3.2157680e-01,  4.1665947e-01, -3.3332862e-02],
        [ 1.2092095e+01,  4.8902116e+00,  1.0991470e+00, ...,
          3.5948776e-02,  9.6325815e-02, -5.5510742e-03],
        ...,
        [ 1.1331301e+01,  5.3314710e+00,  1.6507509e+00, ...,
          3.0972598e-02, -5.0862026e-01,  1.9229789e-01],
        [ 1.2981204e+01, -3.5039918e+00,  1.4301628e-01, ...,
         -6.2189113e-02,  8.5625067e-02,  1.1078746e-01],
        [ 1.1920229e+01,  5.4163823e+00,  1.5074321e+00, ...,
          4.6933506e-02,  8.5574016e-02,  1.7773852e-01]], dtype=float32),
 (7105, 70))

In [12]:
del input_train
del train_data
del train_protein_data
del test_data

## Normalize dimension-reduced gene expression

In [13]:
train_sd = np.std(train_matrix, axis=1).reshape(-1, 1)
train_sd[train_sd == 0] = 1
train_norm = (train_matrix - np.mean(train_matrix, axis=1).reshape(-1, 1)) / train_sd
train_norm = train_norm.astype(np.float32)
del train_matrix
train_norm

array([[ 1.60716133e+01, -2.90949082e+00, -3.20524669e+00, ...,
        -1.69372261e-01,  6.27792254e-02, -2.69252390e-01],
       [ 1.52507648e+01,  2.86405897e+00, -3.97562766e+00, ...,
        -1.70952156e-01, -6.80635393e-01,  1.12298295e-01],
       [ 1.54823751e+01, -2.90800500e+00, -4.68256044e+00, ...,
        -3.84830013e-02, -5.63454535e-03,  2.04880744e-01],
       ...,
       [ 1.50168409e+01, -2.19576740e+00, -4.09990835e+00, ...,
         4.12786543e-01,  5.62575124e-02, -5.57198860e-02],
       [ 1.56578703e+01,  2.82735777e+00, -3.83671165e+00, ...,
        -1.65937737e-01, -1.34040462e-02, -1.28617018e-01],
       [ 1.56788721e+01, -2.89371133e+00, -4.44281006e+00, ...,
         4.05564606e-02, -1.64331034e-01, -5.10188997e-01]], dtype=float32)

In [14]:
test_sd = np.std(test_matrix, axis=1).reshape(-1, 1)
test_sd[test_sd == 0] = 1
test_norm = (test_matrix - np.mean(test_matrix, axis=1).reshape(-1, 1)) / test_sd
test_norm = test_norm.astype(np.float32)
del test_matrix
test_norm

array([[ 1.63436947e+01, -3.68474674e+00,  4.30256784e-01, ...,
         1.72200859e-01, -3.79177600e-01, -6.72205508e-01],
       [ 1.60188847e+01,  3.21042061e+00,  6.55329406e-01, ...,
         4.03425515e-01, -6.65037930e-02, -2.61577349e-02],
       [ 1.62893696e+01,  3.19022512e+00,  2.15013933e+00, ...,
        -3.28458548e-01, -4.58673798e-02, -1.45613313e-01],
       ...,
       [ 1.61357059e+01, -3.69371867e+00,  8.59292626e-01, ...,
        -6.32864982e-02,  1.03970625e-01, -1.29476964e-01],
       [ 1.62148457e+01, -3.63846660e+00,  2.27454320e-01, ...,
         7.76881585e-03, -2.03953460e-01, -4.51042056e-02],
       [ 1.24349804e+01,  3.44128108e+00, -5.92893302e-01, ...,
        -1.15564518e-01,  1.87902585e-01,  1.01284254e+00]], dtype=float32)

## Start training

In [15]:
y_pred = np.zeros((pred_dimx, pred_dimy), dtype=np.float32)
np.random.seed(1000)
for _ in range(5):   
    np.random.shuffle(batches)
    for batch in [batches[:batch_len//2], batches[batch_len//2:]]:   
        if not batch:
            batch = [batches[0]]
#
        kernel = RBF(length_scale=scale)
        krr = KernelRidge(alpha=alpha, kernel=kernel)
        krr.fit(train_norm[feature_obs.batch.isin(batch)], train_gs[gs_obs.batch.isin(batch)])
        if train_gs.shape[1] != y_pred.shape[1]:
            y_pred += (krr.predict(test_norm) @ embedder_mod2.components_)
        else:
            y_pred += krr.predict(test_norm)

## Save prediction

In [16]:
np.clip(y_pred, a_min=0, a_max=None, out=y_pred)
y_pred /= 10
y_pred_df = pd.DataFrame(y_pred, index=test_obs.index, columns=protein_names)
y_pred_df.to_csv(f"{base_dir}/test_predicted_protein_expression (Dengkw).csv")
y_pred_df

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,0.358943,1.432053,1.431793,0.418352,0.417153,0.287139,0.325579,0.656129,2.631083,0.591260,...,0.772705,1.615351,0.444063,0.126417,0.842788,0.336976,0.671961,0.325607,0.559644,1.978624
AAACCCACACCTCTAC-1,0.324878,1.952796,0.587544,0.367211,0.552290,0.431445,0.389398,0.482114,2.473011,0.557783,...,0.697310,0.413600,0.392947,0.102348,0.790729,0.341265,0.824657,0.345110,0.498108,1.258495
AAACCCACACGTCGTG-1,0.358395,2.653279,0.622599,0.355859,0.690353,0.414692,0.370811,0.450563,1.968379,0.650028,...,0.691600,0.634694,0.371962,0.134147,0.737194,0.334021,0.783569,0.310431,0.500080,1.394203
AAACCCACATGTCTAG-1,0.331668,2.572171,0.583418,0.366426,0.609405,0.380850,0.373524,0.462698,2.112783,0.664189,...,0.719484,0.511723,0.390880,0.145161,0.769804,0.342772,0.808538,0.367373,0.559084,1.236230
AAACCCAGTCAAAGTA-1,0.331242,2.051100,0.574184,0.328800,0.507531,0.399767,0.367895,0.472637,2.105166,0.519088,...,0.721337,0.347221,0.403173,0.106990,0.791320,0.333196,0.746540,0.387296,0.523666,1.406195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,0.349812,2.026726,0.597056,0.390493,0.522036,0.437645,0.370438,0.478940,1.807254,0.652756,...,0.775332,0.496026,0.420992,0.132496,0.839300,0.361935,0.735479,0.364261,0.533466,1.329001
TTTGTTGGTGCTCTTC-1,0.317489,2.142422,0.568956,0.377466,0.486348,0.421470,0.321803,0.548029,1.981184,0.712713,...,0.726628,0.616070,0.426028,0.128972,0.803999,0.336149,0.784969,0.381830,0.542410,1.435763
TTTGTTGTCAGCCTTC-1,0.350529,1.516935,0.507039,0.382683,0.382363,0.307320,0.328195,0.599481,2.781341,0.711420,...,0.773593,1.953615,0.454715,0.118741,0.831188,0.354351,0.712193,0.331937,0.452638,1.067295
